In [1]:
import sys
import plotly.graph_objects as go
from pathlib import Path
import pandas as pd
import numpy as np
import re
# --- Data Loading ---

files = [
    'data/prices_round_1_day_-1.csv',
    'data/prices_round_1_day_-2.csv',
    'data/prices_round_1_day_0.csv'
]

prices = []
for file in files:
    try:
        df = pd.read_csv(file, sep=';')
        day_match = re.search(r'day_(-?\d+)', file)
        if day_match:
            df['day'] = int(day_match.group(1))
        prices.append(df)
    except FileNotFoundError:
        print(f"Warning: {file} not found.")

df_total = pd.concat(prices, ignore_index=True)
df_total = df_total.sort_values(by=['day', 'timestamp']).reset_index(drop=True)

In [ ]:

def calculate_book_vwap(df):
    nominal = 0
    volume = 0
    for i in range(1, 4):
        nominal += (df[f'bid_price_{i}'] * df[f'bid_volume_{i}']).fillna(0)
        nominal += (df[f'ask_price_{i}'] * df[f'ask_volume_{i}']).fillna(0)
        volume += df[f'bid_volume_{i}'].fillna(0)
        volume += df[f'ask_volume_{i}'].fillna(0)
    return nominal / volume

def calculate_micro_price(df):
    """
    Calculates the volume-weighted micro-price.
    Uses Level 1 Bid/Ask prices and volumes.
    """
    bid_p, bid_v = df['bid_price_1'], df['bid_vol_1']
    ask_p, ask_v = df['ask_price_1'], df['ask_vol_1']
    
    # Standard Micro-price formula: (Pb * Va + Pa * Vb) / (Vb + Va)
    # This weights the price toward the side with more volume (liquidity)
    micro_price = (bid_p * ask_v + ask_p * bid_v) / (bid_v + ask_v)
    
    # Fallback: if volume is 0 on both sides, use Mid Price
    micro_price = micro_price.fillna((bid_p + ask_p) / 2)
    
    return micro_price

In [66]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.diagnostic import acorr_ljungbox
import plotly.graph_objects as go

# Configuration
days = [-1, -2, 0]
window_size = 300 # Ticks for rolling average/median

for day in days:
    # Filter and copy subset
    subset = df_total[(df_total['product'] == "ASH_COATED_OSMIUM") & (df_total['day'] == day)].copy()
    
    if subset.empty:
        print(f"Skipping day {day}: No data found.")
        continue
    
    # 1. Calculate Micro Price and Handle Mid Price
    subset['mid_price'] = subset['mid_price'].replace(0, np.nan)
    subset['micro_price'] = calculate_micro_price(subset)
    
    # 2. Calculate Rolling Stats
    subset['mid_price_roll_median'] = subset['mid_price'].rolling(window=window_size).median()
    subset['micro_roll_median'] = subset['micro_price'].rolling(window=window_size).median()
    
    # 3. Statistical Analysis Suite (Residuals)
    # Using Mid Price vs Median as the primary noise baseline
    subset['mid_residuals'] = subset['mid_price'] - subset['mid_price_roll_median']
    res_clean = subset['mid_residuals'].dropna()

    if not res_clean.empty:
        # Ljung-Box for Autocorrelation (White Noise Test)
        lb_results = acorr_ljungbox(res_clean, lags=[10], return_df=True)
        lb_pvalue = lb_results['lb_pvalue'].iloc[0]
        
        # Distribution Metrics
        kurt = stats.kurtosis(res_clean)
        skew = stats.skew(res_clean)
        _, norm_p = stats.normaltest(res_clean)

        print(f"\n--- Statistical Suite: Day {day} ---")
        print(f"Ljung-Box (Lag 10) p-value: {lb_pvalue:.5f}")
        print(f"  > Status: {'White Noise' if lb_pvalue > 0.05 else 'Signal Leakage (Autocorrelated)'}")
        print(f"Excess Kurtosis: {kurt:.2f}")
        print(f"  > Distribution: {'Fat-Tailed (Outlier Heavy)' if kurt > 0 else 'Thin-Tailed'}")
        print(f"Normality p-value: {norm_p:.5f}")
        print("-" * 35)

        # Optional: Distribution Plot
        fig_hist = go.Figure(data=[go.Histogram(x=res_clean, nbinsx=50, marker_color='gray')])
        fig_hist.update_layout(title=f"Residual Distribution - Day {day}", template='plotly_white')
        fig_hist.show()

    # 4. Create the Interactive Plotly Figure
    fig = go.Figure()


    fig.add_trace(go.Scatter(x=subset['timestamp'], y=subset['mid_price_roll_median'], 
                             name='Mid Rolling Median', line=dict(color='darkblue', dash='dot')))
    
    # Order Book Levels
    fig.add_trace(go.Scatter(x=subset['timestamp'], y=subset['bid_price_1'], 
                             name='Bid 1', line=dict(color='green', dash='dot', width=1)))
    fig.add_trace(go.Scatter(x=subset['timestamp'], y=subset['ask_price_1'], 
                             name='Ask 1', line=dict(color='red', dash='dot', width=1)))


    fig.add_trace(go.Scatter(x=subset['timestamp'], y=subset['micro_roll_median'], 
                             name='micro_price Rolling Median', line=dict(color='firebrick', dash='dot')))

    # Layout Customization
    fig.update_layout(
        title=f'Market Analysis: ASH_COATED_OSMIUM - Day {day}',
        xaxis_title='Timestamp',
        yaxis_title='Price',
        legend_title='Metrics',
        template='plotly_white',
        hovermode='x unified'
    )
    
    fig.show()


--- Statistical Suite: Day -1 ---
Ljung-Box (Lag 10) p-value: 0.00000
  > Status: Signal Leakage (Autocorrelated)
Excess Kurtosis: 1.36
  > Distribution: Fat-Tailed (Outlier Heavy)
Normality p-value: 0.00000
-----------------------------------



--- Statistical Suite: Day -2 ---
Ljung-Box (Lag 10) p-value: 0.00000
  > Status: Signal Leakage (Autocorrelated)
Excess Kurtosis: 0.98
  > Distribution: Fat-Tailed (Outlier Heavy)
Normality p-value: 0.00000
-----------------------------------



--- Statistical Suite: Day 0 ---
Ljung-Box (Lag 10) p-value: 0.00000
  > Status: Signal Leakage (Autocorrelated)
Excess Kurtosis: 0.96
  > Distribution: Fat-Tailed (Outlier Heavy)
Normality p-value: 0.00000
-----------------------------------


In [67]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.diagnostic import acorr_ljungbox
import plotly.graph_objects as go

# Configuration
days = [-1, -2, 0]
window_size = 3 

for day in days:
    subset = df_total[(df_total['product'] == "ASH_COATED_OSMIUM") & (df_total['day'] == day)].copy()
    if subset.empty: continue
    
    subset['mid_price'] = subset['mid_price'].replace(0, np.nan)
    subset['mid_price_roll_median'] = subset['mid_price'].rolling(window=window_size).median()
    subset['mid_residuals'] = subset['mid_price'] - subset['mid_price_roll_median']
    res_clean = subset['mid_residuals'].dropna()

    if not res_clean.empty:
        # Fit Laplace parameters
        loc_lap, scale_lap = stats.laplace.fit(res_clean)
        
        # Create a range for the theoretical PDF curve
        x_pdf = np.linspace(res_clean.min(), res_clean.max(), 1000)
        y_pdf = stats.laplace.pdf(x_pdf, loc_lap, scale_lap)

        # Calculate metrics
        kurt = stats.kurtosis(res_clean)
        ks_stat, laplace_p = stats.kstest(res_clean, 'laplace', args=(loc_lap, scale_lap))

        print(f"\n--- Analysis: Day {day} ---")
        print(f"Excess Kurtosis: {kurt:.2f} (Target for Laplace is 3.0)")
        print(f"Laplace KS p-value: {laplace_p:.5f}")

        # Distribution Plot with PDF Overlay
        fig_hist = go.Figure()

        # Actual Residuals Histogram (Normalized to show density)
        fig_hist.add_trace(go.Histogram(
            x=res_clean, 
            histnorm='probability density', 
            name='Actual Residuals',
            marker_color='rgba(100, 100, 100, 0.6)'
        ))

        # Theoretical Laplace PDF
        fig_hist.add_trace(go.Scatter(
            x=x_pdf, 
            y=y_pdf, 
            mode='lines', 
            name='Theoretical Laplace',
            line=dict(color='red', width=2)
        ))

        fig_hist.update_layout(
            title=f"Residual Density vs Laplace Fit - Day {day} (Kurtosis: {kurt:.2f})",
            xaxis_title="Residual Value (Ticks)",
            yaxis_title="Density",
            template='plotly_white'
        )
        fig_hist.show()


--- Analysis: Day -1 ---
Excess Kurtosis: 6.98 (Target for Laplace is 3.0)
Laplace KS p-value: 0.00000



--- Analysis: Day -2 ---
Excess Kurtosis: 6.72 (Target for Laplace is 3.0)
Laplace KS p-value: 0.00000



--- Analysis: Day 0 ---
Excess Kurtosis: 7.10 (Target for Laplace is 3.0)
Laplace KS p-value: 0.00000


In [69]:
import numpy as np
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.diagnostic import acorr_ljungbox
import plotly.graph_objects as go

def hampel_filter_pandas(series, window_size=17, n_sigma=3):
    """
    Fast Hampel Filter implementation using pandas rolling windows.
    
    Parameters:
    - series: pd.Series, the price data
    - window_size: int, size of the window (should be odd)
    - n_sigma: float, threshold for outlier detection (standard is 3)
    
    Returns:
    - filtered_series: pd.Series with outliers replaced by rolling median
    - outliers: pd.Series (boolean mask) indicating where outliers were found
    """
    # 1. Calculate rolling median
    L = window_size // 2
    rolling_median = series.rolling(window=window_size, center=True).median()
    
    # 2. Calculate rolling Median Absolute Deviation (MAD)
    # MAD = median(|x_i - median(x)|)
    rolling_mad = series.rolling(window=window_size, center=True).apply(
        lambda x: np.median(np.abs(x - np.median(x))), raw=True
    )
    
    # 3. Scale MAD to be consistent with standard deviation (sigma)
    # For a normal distribution, sigma is approx 1.4826 * MAD
    constant = 1.4826
    rolling_sigma = constant * rolling_mad
    
    # 4. Identify outliers
    # Deviation from median > n_sigma * sigma
    outlier_mask = np.abs(series - rolling_median) > (n_sigma * rolling_sigma)
    
    # 5. Replace outliers with the rolling median
    filtered_series = series.copy()
    filtered_series[outlier_mask] = rolling_median[outlier_mask]
    
    return filtered_series, outlier_mask

# --- Main Analysis Loop ---
days = [-1, -2, 0]
window_size = 17 
n_sigma_threshold = 3 # Adjust to 2.0 for tighter filtering of the bid-ask bounce

for day in days:
    # Filter for the specific product and day
    subset = df_total[(df_total['product'] == "ASH_COATED_OSMIUM") & (df_total['day'] == day)].copy()
    
    if subset.empty:
        continue
    
    # Clean data
    subset['mid_price'] = subset['mid_price'].replace(0, np.nan)
    
    # Apply Hampel Filter
    # We apply this to the mid_price to find the 'Fair Value'
    subset['hampel_mid'], outliers = hampel_filter_pandas(
        subset['mid_price'], 
        window_size=window_size, 
        n_sigma=n_sigma_threshold
    )
    
    # Calculate Residuals (Noise)
    # Noise = Actual Price - Filtered Fair Value
    subset['hampel_residuals'] = subset['mid_price'] - subset['hampel_mid']
    res_clean = subset['hampel_residuals'].dropna()

    if not res_clean.empty:
        # Statistical Tests
        # 1. Ljung-Box for Signal Leakage
        lb_p = acorr_ljungbox(res_clean, lags=[10])['lb_pvalue'].iloc[0]
        
        # 2. Laplace Fit Test
        loc_lap, scale_lap = stats.laplace.fit(res_clean)
        _, laplace_p = stats.kstest(res_clean, 'laplace', args=(loc_lap, scale_lap))
        
        # 3. Kurtosis (Targeting ~3.0 for Laplace)
        kurt = stats.kurtosis(res_clean)

        print(f"\n--- Hampel Analysis: Day {day} ---")
        print(f"Outliers detected: {outliers.sum()} ({100*outliers.sum()/len(subset):.2f}%)")
        print(f"Ljung-Box p: {lb_p:.5f} ({'White Noise' if lb_p > 0.05 else 'Signal Leakage'})")
        print(f"Laplace KS p: {laplace_p:.5f}")
        print(f"Excess Kurtosis: {kurt:.2f}")
        print("-" * 35)

        # Plot 1: The Filtering Effect
        fig_price = go.Figure()
        fig_price.add_trace(go.Scatter(x=subset['timestamp'], y=subset['mid_price'], name='Raw Mid Price', line=dict(color='lightgray')))
        fig_price.add_trace(go.Scatter(x=subset['timestamp'], y=subset['hampel_mid'], name='Hampel Fair Value', line=dict(color='blue')))
        fig_price.add_trace(go.Scatter(x=subset['timestamp'][outliers], y=subset['mid_price'][outliers], 
                                      mode='markers', name='Outliers (Storm)', marker=dict(color='red', size=8)))
        fig_price.update_layout(title=f"Hampel Filter Price Action - Day {day}", template='plotly_white')
        fig_price.show()

        # Plot 2: Residual Distribution
        fig_dist = go.Figure()
        fig_dist.add_trace(go.Histogram(x=res_clean, histnorm='probability density', name='Hampel Residuals', marker_color='gray'))
        
        # Overlay Laplace Fit
        x_range = np.linspace(res_clean.min(), res_clean.max(), 100)
        fig_dist.add_trace(go.Scatter(x=x_range, y=stats.laplace.pdf(x_range, loc_lap, scale_lap), name='Laplace Fit', line=dict(color='red')))
        
        fig_dist.update_layout(title=f"Residual Distribution - Day {day}", template='plotly_white')
        fig_dist.show()


--- Hampel Analysis: Day -1 ---
Outliers detected: 1577 (15.77%)
Ljung-Box p: 0.96368 (White Noise)
Laplace KS p: 0.00000
Excess Kurtosis: 8.55
-----------------------------------



--- Hampel Analysis: Day -2 ---
Outliers detected: 1587 (15.87%)
Ljung-Box p: 0.47564 (White Noise)
Laplace KS p: 0.00000
Excess Kurtosis: 8.52
-----------------------------------



--- Hampel Analysis: Day 0 ---
Outliers detected: 1600 (16.00%)
Ljung-Box p: 0.58547 (White Noise)
Laplace KS p: 0.00000
Excess Kurtosis: 8.67
-----------------------------------


In [70]:
import numpy as np
import scipy.stats as stats
from statsmodels.sandbox.stats.runs import runstest_1samp

def run_diagnostic_suite(res_clean, outlier_mask):
    results = {}
    
    # --- 1. Student's t-Distribution Fit ---
    # Purpose: Model the heavy tails (Kurtosis ~8.6)
    df_t, loc_t, scale_t = stats.t.fit(res_clean)
    ks_t, p_t = stats.kstest(res_clean, 't', args=(df_t, loc_t, scale_t))
    results['t_fit'] = {'nu': df_t, 'p_value': p_t}

    # --- 2. Runs Test (Wald-Wolfowitz) ---
    # Purpose: Determine if 'Storms' (outliers) are clustered in time or random.
    # We test the outlier_mask (True/False). 
    # If p < 0.05, outliers are CLUSTERED (Regime behavior confirmed).
    z_stat, p_runs = runstest_1samp(outlier_mask.astype(int), cutoff='mean')
    results['runs_test'] = p_runs

    # --- 3. Variance Ratio Test (Simplified) ---
    # Purpose: Confirm Mean Reversion. 
    # Ratio = Var(k-period return) / (k * Var(1-period return))
    # Ratio < 1 implies Mean Reversion.
    k = 5
    var_1 = np.var(res_clean)
    # Calculate k-period variance (rolling sum of residuals over k ticks)
    var_k = np.var(res_clean.rolling(window=k).sum().dropna())
    v_ratio = var_k / (k * var_1)
    results['v_ratio'] = v_ratio
    
    return results

# --- Integration Example ---
for day in days:
    # ... (Assuming Hampel filter code from previous step has run) ...
    
    if not res_clean.empty:
        diagnostics = run_diagnostic_suite(res_clean, outliers)
        
        print(f"\n==== Advanced Diagnostics: Day {day} ====")
        
        # Interpretation of t-distribution
        print(f"Student's t nu (Degrees of Freedom): {diagnostics['t_fit']['nu']:.2f}")
        print(f"Student's t KS p-value: {diagnostics['t_fit']['p_value']:.5f}")
        
        # Interpretation of Runs Test
        p_runs = diagnostics['runs_test']
        print(f"Runs Test p-value: {p_runs:.5f}")
        print(f" > Status: {'Regimes are CLUSTERED (Predictable Storms)' if p_runs < 0.05 else 'Outliers are Random'}")
        
        # Interpretation of Variance Ratio
        vr = diagnostics['v_ratio']
        print(f"Variance Ratio (k=5): {vr:.3f}")
        print(f" > Status: {'STRONG Mean Reversion' if vr < 0.8 else 'Random Walk / Weak Reversion'}")
        print("="*40)


==== Advanced Diagnostics: Day -1 ====
Student's t nu (Degrees of Freedom): 2.12
Student's t KS p-value: 0.00000
Runs Test p-value: 0.00000
 > Status: Regimes are CLUSTERED (Predictable Storms)
Variance Ratio (k=5): 1.006
 > Status: Random Walk / Weak Reversion

==== Advanced Diagnostics: Day -2 ====
Student's t nu (Degrees of Freedom): 2.12
Student's t KS p-value: 0.00000
Runs Test p-value: 0.00000
 > Status: Regimes are CLUSTERED (Predictable Storms)
Variance Ratio (k=5): 1.006
 > Status: Random Walk / Weak Reversion

==== Advanced Diagnostics: Day 0 ====
Student's t nu (Degrees of Freedom): 2.12
Student's t KS p-value: 0.00000
Runs Test p-value: 0.00000
 > Status: Regimes are CLUSTERED (Predictable Storms)
Variance Ratio (k=5): 1.006
 > Status: Random Walk / Weak Reversion


In [71]:
def test_signal_edge(subset, threshold_ticks=3):
    """
    Checks if the price actually moves back toward the mean 
    AFTER hitting a specific residual threshold.
    """
    # 1. Identify where residual is extreme
    signal_mask = subset['hampel_residuals'].abs() >= threshold_ticks
    
    # 2. Look at the return over the NEXT 5 ticks
    subset['future_return'] = subset['mid_price'].shift(-5) - subset['mid_price']
    
    # 3. Calculate expected return when signal is active
    # We multiply by -sign of residual because we expect reversion
    edge = (subset['future_return'] * -np.sign(subset['hampel_residuals']))[signal_mask].mean()
    
    return edge

# Run for thresholds 1, 2, 3, 4
for t in [1, 2, 3, 4]:
    print(f"Edge for {t} tick threshold: {test_signal_edge(subset, t):.4f}")

Edge for 1 tick threshold: 5.3012
Edge for 2 tick threshold: 6.7877
Edge for 3 tick threshold: 7.1397
Edge for 4 tick threshold: 7.4318


In [72]:
def test_alpha_durability(subset, horizons=[1, 3, 5, 10, 20]):
    print(f"{'Horizon':<10} | {'IC (Correlation)':<15} | {'Edge (Ticks)':<10}")
    print("-" * 45)
    
    for h in horizons:
        # Forward return
        fwd_return = subset['mid_price'].shift(-h) - subset['mid_price']
        
        # Calculate Information Coefficient (Correlation between signal and outcome)
        valid_idx = ~(subset['hampel_residuals'].isna() | fwd_return.isna())
        ic, _ = stats.spearmanr(subset['hampel_residuals'][valid_idx], fwd_return[valid_idx])
        
        # Calculate Edge (Sign-adjusted return)
        edge = (fwd_return * -np.sign(subset['hampel_residuals'])).mean()
        
        print(f"{h:<10} | {ic:<15.4f} | {edge:<10.4f}")

# Run this to find your 'Sweet Spot' horizon
test_alpha_durability(subset)

Horizon    | IC (Correlation) | Edge (Ticks)
---------------------------------------------
1          | -0.4908         | 0.8077    
3          | -0.4953         | 0.8110    
5          | -0.4892         | 0.8198    
10         | -0.4703         | 0.8312    
20         | -0.4433         | 0.7947    


In [73]:
def test_trade_capacity(subset, thresholds=[1, 2, 3, 4]):
    print(f"{'Threshold':<10} | {'Trades/Day':<12} | {'Total Profit (Ticks)':<15}")
    print("-" * 45)
    
    # Forward return at the 10-tick 'Sweet Spot'
    fwd_10 = subset['mid_price'].shift(-10) - subset['mid_price']
    
    for t in thresholds:
        # Identify entries (Signal > Threshold)
        # We use 'diff' to ensure we only count the 'Entry' tick, not every tick during the storm
        entries = (subset['hampel_residuals'].abs() >= t) & (subset['hampel_residuals'].abs().shift(1) < t)
        
        num_trades = entries.sum()
        avg_profit = (fwd_10 * -np.sign(subset['hampel_residuals']))[entries].mean()
        total_profit = num_trades * avg_profit
        
        print(f"{t:<10} | {num_trades:<12} | {total_profit:<15.2f}")

test_trade_capacity(subset)

Threshold  | Trades/Day   | Total Profit (Ticks)
---------------------------------------------
1          | 1214         | 6940.72        
2          | 993          | 6923.47        
3          | 934          | 6805.29        
4          | 877          | 6673.61        
